# Fabric Link migration checks: attendee edition
Run in a **Python 3.12** notebook. Attach a separate writable lakehouse for snapshots.
The notebook resolves the workspace names configured below, then checks shortcut targets, SQL tables, and Direct Lake model sources in those workspaces.
Normal runs read Fabric items and write diagnostic snapshots. Flag-controlled optional cells can refresh
SQL metadata or repair shortcut targets when explicitly enabled. Keep the consumer lakehouse
and snapshot lakehouse through migration. No cell rebinds models or changes permissions.

Run before migration with `MODE = "before"`, then rerun after migration with `MODE = "after"`.
Names are optional overrides when multiple candidates exist; no row numbers or item IDs are needed.
The baseline is saved once and never overwritten. Use a distinct `MIGRATION_RUN_NAME` for each migration. Keep the same run name and snapshot lakehouse for the before and after runs. A missing baseline blocks comparison only.

Table tests use the **consumer SQL analytics endpoint**, not direct Delta reads.
Model tests query one discovered table per relevant Direct Lake model. They do not refresh models,
prove freshness, validate every measure/RLS role, or guarantee report-user permissions.
Confirm the Low-latency mode flag manually in Power Apps. Failed checks and permission errors
are shown in the report; missing information is not counted as success.

References: [shortcut API](https://learn.microsoft.com/en-us/rest/api/fabric/core/onelake-shortcuts/list-shortcuts),
[Direct Lake sources](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.directlake.html).

## Setup and run order
1. Import this notebook into Microsoft Fabric. Use a Python 3.12 environment compatible with the original notebook. Attach a separate writable default lakehouse for snapshots, which must survive the migration.
2. Run the configure, package-install and Python-restart cells. After restart, run Settings and continue from there.
3. Fill in the source workspace name below. Select the consumer workspace and optionally extra model workspaces. The notebook inventories items only in these selected workspaces. Workspace-name resolution lists accessible workspace names; it does not scan all their items.
4. Run with `MODE = "before"` and confirm a complete baseline was saved before migrating. Perform the migration separately. This notebook does not migrate Fabric Link itself.
5. Switch to `MODE = "after"` and rerun Settings, discovery, checks and report using the same snapshot folder. Review every changed, broken or unverified result.

Required access: read access to selected Fabric items, SQL endpoint/table access, and semantic-model access sufficient for metadata and DAX queries. Optional repair and refresh need additional permissions. Permission failures remain unverified results.

**Sharing:** this distributed copy has no execution outputs, credentials or attached lakehouse identifiers. Your runs generate workspace/item names and IDs, schema metadata, shortcut definitions and diagnostics in outputs and snapshots. Review and clear those before sharing your executed copy. Runtime authentication uses your Fabric session.


In [ ]:
%%configure -f
{
    "vCores": 2
}


In [ ]:
%pip install --upgrade semantic-link-labs==0.17.1 requests


In [ ]:
notebookutils.session.restartPython()


## Settings: workspaces and migration run
Edit the next cell before running discovery. Source is the workspace containing the linked Dataverse lakehouse. Consumer is the workspace containing the stable lakehouse with shortcuts. Additional model workspaces are optional. This checks one source/consumer pair per run. Use a new run name for another pair, but keep the same name across its before/after runs.


In [ ]:
# REQUIRED: replace with your Fabric workspace display name.
SOURCE_WORKSPACE_NAME = ""  # Workspace containing the linked Dataverse source lakehouse

# None uses the notebook's current workspace. Set a name for a different workspace.
CONSUMER_WORKSPACE_NAME = None
# Only these additional workspaces are scanned for semantic models.
MODEL_WORKSPACE_NAMES = []  # Example: ["Reporting workspace"]

MODE = "before"  # "before" to capture a baseline, "after" to compare
MIGRATION_RUN_NAME = "migration_01"  # Same for before/after. New name for another migration.
CONSUMER_LAKEHOUSE_NAME = None  # Set if automatic selection is ambiguous
SOURCE_LAKEHOUSE_NAME = None    # Set to the replacement source after migration if ambiguous
SNAPSHOT_DIR = f"/lakehouse/default/Files/fabric_link_migration_checks/attendee/{MIGRATION_RUN_NAME}"
LOW_LATENCY_FLAG_OBSERVED = None  # True only after manual confirmation in Power Apps

# Optional actions. Default runs read items and write diagnostic snapshots only.
RUN_SHORTCUT_REPAIR = False    # True previews proposed shortcut target changes
APPLY_SHORTCUT_REPAIR = False  # True with RUN_SHORTCUT_REPAIR applies reviewed changes
RUN_SQL_REFRESH = False        # Requests consumer SQL endpoint metadata refresh
RUN_SQL_DIAGNOSTICS = False    # Extra SQL connection/table probes

if not isinstance(SOURCE_WORKSPACE_NAME, str) or not SOURCE_WORKSPACE_NAME.strip():
    raise ValueError("Set SOURCE_WORKSPACE_NAME to your linked-source workspace name.")
if CONSUMER_WORKSPACE_NAME is not None and (not isinstance(CONSUMER_WORKSPACE_NAME, str) or not CONSUMER_WORKSPACE_NAME.strip()):
    raise ValueError("CONSUMER_WORKSPACE_NAME must be None or a nonempty workspace name.")
if not isinstance(MODEL_WORKSPACE_NAMES, list) or any(not isinstance(v, str) or not v.strip() for v in MODEL_WORKSPACE_NAMES):
    raise ValueError("MODEL_WORKSPACE_NAMES must be a list of nonempty workspace names.")
if not isinstance(MIGRATION_RUN_NAME, str) or not MIGRATION_RUN_NAME or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in MIGRATION_RUN_NAME):
    raise ValueError("Use letters, numbers, underscores or hyphens for MIGRATION_RUN_NAME.")


In [ ]:
import json
import time
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit
import requests
import pandas as pd
import sempy.fabric as fabric
import sempy_labs.directlake as dl
from importlib.metadata import version
print("Semantic Link Labs:", version("semantic-link-labs"))
table_checks_current = False  # Discovery invalidates any earlier table-check results.

if MODE not in {"before", "after"}:
    raise ValueError('MODE must be "before" or "after".')
CURRENT_WORKSPACE_ID = str(notebookutils.runtime.context["currentWorkspaceId"])
diagnostics = []
checks = []

def check(area, name, status, detail):
    checks.append({"area": area, "name": name, "status": status, "detail": detail})

def problem(operation, exc):
    diagnostics.append({"operation": operation, "error_type": type(exc).__name__, "error": str(exc)})
    check("Discovery", operation, "couldn't verify", str(exc))

def api_get(path):
    if path.startswith("https://"):
        parsed = urlsplit(path)
        if parsed.netloc != "api.fabric.microsoft.com":
            raise ValueError("Unexpected continuation host")
        path = parsed.path + ("?" + parsed.query if parsed.query else "")
    url = "https://api.fabric.microsoft.com/" + path.lstrip("/")
    for attempt in range(3):
        response = requests.get(url, headers={"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")}, timeout=60)
        if response.status_code == 429 and attempt < 2:
            delay = response.headers.get("Retry-After", "5")
            time.sleep(min(int(delay) if delay.isdigit() else 5, 15))
            continue
        if not response.ok:
            raise RuntimeError(f"HTTP {response.status_code}: {response.text[:1500]}")
        return response.json()

def list_all(path):
    rows = []
    while path:
        body = api_get(path)
        rows.extend(body.get("value", []))
        path = body.get("continuationUri")
    return rows

def safely(operation, call, fallback=None):
    try:
        return call()
    except Exception as exc:
        problem(operation, exc)
        return fallback

workspaces = safely("List workspaces", lambda: list_all("/v1/workspaces"), [])
def workspace_by_name(name):
    matches = [w for w in workspaces if w["displayName"].casefold() == name.casefold()]
    if len(matches) != 1:
        raise ValueError(f"Expected one accessible workspace named {name!r}; found {len(matches)}.")
    return matches[0]["id"]

source_ws = safely("Find source workspace", lambda: workspace_by_name(SOURCE_WORKSPACE_NAME))
consumer_ws = CURRENT_WORKSPACE_ID if CONSUMER_WORKSPACE_NAME is None else safely(
    "Find consumer workspace", lambda: workspace_by_name(CONSUMER_WORKSPACE_NAME))
snapshot_folder = Path(SNAPSHOT_DIR)
baseline_path = snapshot_folder / "before.json"
before = None
if MODE == "after":
    def load_baseline():
        value = json.loads(baseline_path.read_text(encoding="utf-8"))
        if value.get("format_version") != 2:
            raise ValueError("This baseline uses a different notebook format.")
        return value
    before = safely("Load baseline", load_baseline)

items = {}
def workspace_items(ws):
    if ws not in items:
        items[ws] = safely(f"List items in {ws}", lambda: list_all(f"/v1/workspaces/{ws}/items"), [])
    return items[ws]
model_workspaces = {safely(f"Find model workspace: {name}", lambda name=name: workspace_by_name(name))
                    for name in MODEL_WORKSPACE_NAMES}
selected_workspaces = ({source_ws, consumer_ws} | model_workspaces) - {None}
for ws in selected_workspaces:
    workspace_items(ws)
display(pd.DataFrame([{"workspace_id": ws, "name": i["displayName"], "id": i["id"], "type": i["type"]}
                      for ws, values in items.items() for i in values if i["type"] in {"Lakehouse", "SQLEndpoint", "SemanticModel"}]))

shortcut_cache = {}
def shortcuts_for(ws, item):
    key = (ws, item)
    if key not in shortcut_cache:
        shortcut_cache[key] = safely(f"List shortcuts for {item}",
            lambda: list_all(f"/v1/workspaces/{ws}/items/{item}/shortcuts"))
    return shortcut_cache[key]

consumer = None
consumer_candidates = []
if consumer_ws:
    lakes = [i for i in workspace_items(consumer_ws) if i["type"] == "Lakehouse"]
    if MODE == "after" and before and before.get("consumer"):
        old = before["consumer"]
        if old["workspace_id"] == consumer_ws:
            consumer_candidates = [i for i in lakes if i["id"] == old["id"]]
        if not consumer_candidates:
            check("Identity", "Consumer lakehouse", "broken", "Baseline consumer ID no longer exists in the selected workspace.")
    elif CONSUMER_LAKEHOUSE_NAME:
        consumer_candidates = [i for i in lakes if i["displayName"].casefold() == CONSUMER_LAKEHOUSE_NAME.casefold()]
    else:
        for lake in lakes:
            targets = shortcuts_for(consumer_ws, lake["id"])
            if targets and source_ws and any(s.get("target", {}).get("oneLake", {}).get("workspaceId") == source_ws for s in targets):
                consumer_candidates.append(lake)
        # A lone lakehouse is a usable candidate even if shortcut discovery failed/returned empty.
        if not consumer_candidates and len(lakes) == 1:
            consumer_candidates = lakes
    if len(consumer_candidates) == 1:
        consumer = {"workspace_id": consumer_ws, "id": consumer_candidates[0]["id"], "name": consumer_candidates[0]["displayName"]}
        check("Identity", "Consumer lakehouse", "working", consumer["name"])
    elif not consumer:
        check("Discovery", "Consumer selection", "couldn't verify", "Set CONSUMER_LAKEHOUSE_NAME to one of the lakehouse names above; discovery found zero or multiple candidates.")

consumer_shortcuts = shortcuts_for(consumer_ws, consumer["id"]) if consumer else None
source_lakes = [i for i in workspace_items(source_ws) if i["type"] == "Lakehouse"] if source_ws else []
source_ids = {i["id"] for i in source_lakes}
live_target_ids = {s.get("target", {}).get("oneLake", {}).get("itemId") for s in (consumer_shortcuts or [])
                   if s.get("target", {}).get("oneLake", {}).get("workspaceId") == source_ws}
if SOURCE_LAKEHOUSE_NAME:
    source_candidates = [i for i in source_lakes if i["displayName"].casefold() == SOURCE_LAKEHOUSE_NAME.casefold()]
else:
    source_candidates = [i for i in source_lakes if i["id"] in live_target_ids]
    if not source_candidates and len(source_lakes) == 1:
        source_candidates = source_lakes
source = None
if len(source_candidates) == 1:
    source = {"workspace_id": source_ws, "id": source_candidates[0]["id"], "name": source_candidates[0]["displayName"]}
    check("Identity", "Source lakehouse", "working", source["name"])
else:
    check("Discovery", "Source selection", "couldn't verify", "No unique source found. Set SOURCE_LAKEHOUSE_NAME if needed.")
if consumer_shortcuts is not None:
    if not consumer_shortcuts:
        check("Shortcuts", "Consumer inventory", "couldn't verify", "No shortcuts returned. SQL table checks will still run; verify the selected lakehouse.")
    for s in consumer_shortcuts:
        target = s.get("target", {}).get("oneLake", {})
        if target.get("workspaceId") == source_ws:
            exists = target.get("itemId") in source_ids
            inventory_failed = any(d["operation"] == f"List items in {source_ws}" for d in diagnostics)
            status = "couldn't verify" if inventory_failed else "working" if exists else "broken"
            detail = "Source item inventory failed." if inventory_failed else "Target item exists; data read checked separately." if exists else "Target points to a lakehouse ID absent from the source workspace."
            check("Shortcuts", f"{s['path']}/{s['name']}", status, detail)
display(pd.DataFrame([{"role": "source", **(source or {})}, {"role": "consumer", **(consumer or {})}]))
display(pd.json_normalize(consumer_shortcuts or []))


## Optional maintenance and diagnostics — controlled by Settings flags
The next three cells run normally; their flags determine whether they take action.
Run all with all flags False skips repair, refresh, and extra diagnostics.
To use an action, change its flag in Settings and run Settings before its cell.
For repair preview use RUN_SHORTCUT_REPAIR=True and APPLY_SHORTCUT_REPAIR=False.
After reviewing, enable both in Settings, rerun Settings, then run repair manually.
Return the flags in Settings to False afterward.
Each action also resets its flags in memory; it does not rewrite Settings text.

Use the refresh when Lakehouse preview works but SQL cannot see the tables.
This refresh acts on the consumer SQL endpoint, not the Dataverse sync engine.
It cannot force upstream export to complete. Record forced refresh separately from
natural freshness observations. After repair, rerun discovery, table checks and the
after report using the original baseline. Only before migration, use a new folder
if the existing baseline was incomplete.

In [ ]:
# OPTIONAL: bulk repair. Controlled only by the flags in Settings.
def repair_shortcuts():
    if MODE != "after" or not before or not source or not consumer:
        raise ValueError("Run after-mode discovery with the saved baseline first.")
    old_source = before["source"]
    if source["id"] == old_source["id"]:
        raise ValueError("Replacement source must have a different item ID.")
    if (consumer["workspace_id"], consumer["id"]) != (before["consumer"]["workspace_id"], before["consumer"]["id"]):
        raise ValueError("Consumer identity must match the baseline.")
    endpoint = f"/v1/workspaces/{consumer['workspace_id']}/items/{consumer['id']}/shortcuts"
    live = list_all(endpoint)  # fresh inventory; do not use cached discovery
    baseline_map = {(s["path"].strip("/"), s["name"]): s for s in before["shortcuts"]}
    repairs = []
    for shortcut in live:
        target = shortcut.get("target", {}).get("oneLake", {})
        if (target.get("workspaceId"), target.get("itemId")) != (old_source["workspace_id"], old_source["id"]):
            continue
        key = (shortcut["path"].strip("/"), shortcut["name"])
        saved = baseline_map.get(key)
        if not saved or saved.get("target", {}).get("oneLake") != target:
            raise ValueError(f"Shortcut changed since baseline: {key}. Review before repair.")
        repairs.append({"path": key[0], "name": key[1], "target": {"oneLake": {
            "workspaceId": source["workspace_id"], "itemId": source["id"], "path": target["path"]}}})
    display(pd.DataFrame([{"location": r["path"], "shortcut": r["name"],
        "source_path": r["target"]["oneLake"]["path"], "new_source_id": source["id"]} for r in repairs]))
    print(f"{len(repairs)} stale shortcuts found. Apply enabled: {APPLY_SHORTCUT_REPAIR}")
    if APPLY_SHORTCUT_REPAIR and repairs:
        # Save the exact current definitions before the first update.
        repair_backup = Path(SNAPSHOT_DIR) / ("shortcut_repair_backup_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
        repair_backup.write_text(json.dumps(live, indent=2), encoding="utf-8")
        print("Backup:", str(repair_backup))
        for payload in repairs:
            for attempt in range(4):
                response = requests.post("https://api.fabric.microsoft.com" + endpoint,
                    params={"shortcutConflictPolicy": "OverwriteOnly"}, json=payload,
                    headers={"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")}, timeout=60)
                if response.status_code != 429 or attempt == 3:
                    break
                time.sleep(min(int(response.headers.get("Retry-After", "5")), 60))
            response.raise_for_status()
            if response.status_code != 200:
                raise RuntimeError(f"Unexpected shortcut update status: {response.status_code}")
            print("Updated:", payload["path"] + "/" + payload["name"])
        verified = {(s["path"].strip("/"), s["name"]): s for s in list_all(endpoint)}
        for payload in repairs:
            actual = verified.get((payload["path"], payload["name"]), {}).get("target", {}).get("oneLake")
            if actual != payload["target"]["oneLake"]:
                raise RuntimeError("Target verification failed: " + payload["name"])
        shortcut_cache.clear()
        # Update the in-memory inventory so a later snapshot cannot retain old targets.
        globals()["consumer_shortcuts"] = list(verified.values())
        globals()["table_checks_current"] = False
        print("Targets verified. Rerun discovery, then SQL checks and the final after report.")
        print("If SQL metadata lags, manually run the optional endpoint refresh cell first.")
    elif not APPLY_SHORTCUT_REPAIR:
        print("Preview only. Enable APPLY_SHORTCUT_REPAIR in Settings, rerun Settings, then run this cell manually.")

if RUN_SHORTCUT_REPAIR:
    try:
        repair_shortcuts()
    finally:
        RUN_SHORTCUT_REPAIR = False
        APPLY_SHORTCUT_REPAIR = False
else:
    print("Optional repair skipped. Enable RUN_SHORTCUT_REPAIR in Settings for a preview.")


In [ ]:
# OPTIONAL: SQL endpoint metadata refresh. Enable its flag in Settings.
if RUN_SQL_REFRESH:
    try:
        import sempy_labs as labs
        if not consumer:
            raise ValueError("Run discovery and select a consumer first.")
        refresh_result = labs.refresh_sql_endpoint_metadata(
            item=consumer["id"], type="Lakehouse", workspace=consumer["workspace_id"]
        )
        display(refresh_result)
        table_checks_current = False
        print("Explicit endpoint refresh requested. Recheck SQL reads; do not count this as natural propagation.")
    finally:
        RUN_SQL_REFRESH = False
else:
    print("Optional refresh skipped: RUN_SQL_REFRESH is False.")


In [ ]:
# OPTIONAL: connection identity and known shortcut table probes.
if RUN_SQL_DIAGNOSTICS:
    try:
        if not consumer:
            raise ValueError("Run discovery and select a consumer first.")
        with notebookutils.data.connect_to_artifact(consumer["id"], consumer["workspace_id"], "Lakehouse") as connection:
            display(connection.query("SELECT DB_NAME() AS ConnectedDatabase, (SELECT COUNT(*) FROM sys.tables) AS VisibleTableCount", visualize_as_table=False))
            for shortcut in consumer_shortcuts or []:
                parts = shortcut["path"].strip("/").split("/")
                if parts[0] != "Tables" or len(parts) not in (1, 2):
                    continue  # schema-folder shortcuts need explicit table names
                schema = "dbo" if len(parts) == 1 else parts[1]
                name = shortcut["name"]
                quote = lambda value: "[" + value.replace("]", "]]") + "]"
                sql = "SELECT COUNT_BIG(*) AS SampleRows FROM (SELECT TOP (1) 1 AS Probe FROM " + quote(schema) + "." + quote(name) + ") AS sample"
                try:
                    print(f"{schema}.{name}")
                    display(connection.query(sql, visualize_as_table=False))
                except Exception as exc:
                    print(f"Probe failed: {exc}")
    finally:
        RUN_SQL_DIAGNOSTICS = False
else:
    print("Optional diagnostics skipped: RUN_SQL_DIAGNOSTICS is False.")


In [ ]:
# Discover tables from SQL metadata; missing shortcut inventory does not stop reads.
table_checks_current = False
def query_frame(connection, sql):
    result = connection.query(sql, max_rows=-1, visualize_as_table=False)
    if isinstance(result, pd.DataFrame):
        return result
    if isinstance(result, list):
        frames = [value for value in result if isinstance(value, pd.DataFrame)]
        if len(frames) == 1:
            return frames[0]
    raise RuntimeError(
        f"SQL returned {type(result).__name__}, not one DataFrame. Query: {sql}"
    )

def metadata_frame(connection, sql, expected_columns):
    # JSON returns a payload even when no metadata rows are visible.
    raw = connection.query(sql + " FOR JSON PATH", max_rows=-1, visualize_as_table=False)
    if raw is None:
        print("Metadata query returned None; continuing with known shortcut table probes.")
        return pd.DataFrame(columns=expected_columns)
    payload = raw
    if isinstance(raw, list):
        frames = [value for value in raw if isinstance(value, pd.DataFrame)]
        if len(frames) == 1:
            payload = frames[0]
    if not isinstance(payload, pd.DataFrame):
        raise RuntimeError(f"Unexpected metadata result: {type(raw).__name__}")
    if payload.empty:
        return pd.DataFrame(columns=expected_columns)
    text = "".join(str(value) for value in payload.iloc[:, 0])
    return pd.DataFrame(json.loads(text), columns=expected_columns)

tables = []
if consumer:
    try:
        with notebookutils.data.connect_to_artifact(consumer["id"], consumer["workspace_id"], "Lakehouse") as connection:
            identity = query_frame(connection, "SELECT DB_NAME() AS ConnectedDatabase, (SELECT COUNT(*) FROM sys.tables) AS VisibleTableCount")
            display(identity)
            catalog = metadata_frame(connection, "SELECT s.name AS TABLE_SCHEMA, t.name AS TABLE_NAME FROM sys.tables t JOIN sys.schemas s ON t.schema_id = s.schema_id WHERE t.is_ms_shipped = 0 ORDER BY s.name, t.name", ["TABLE_SCHEMA", "TABLE_NAME"])
            columns = metadata_frame(connection, "SELECT s.name AS TABLE_SCHEMA, t.name AS TABLE_NAME, c.name AS COLUMN_NAME, ty.name AS DATA_TYPE, c.is_nullable AS IS_NULLABLE, c.column_id AS ORDINAL_POSITION, c.max_length AS MAX_LENGTH_BYTES, c.precision AS NUMERIC_PRECISION, c.scale AS NUMERIC_SCALE FROM sys.tables t JOIN sys.schemas s ON t.schema_id = s.schema_id JOIN sys.columns c ON t.object_id = c.object_id JOIN sys.types ty ON c.user_type_id = ty.user_type_id WHERE t.is_ms_shipped = 0 ORDER BY s.name, t.name, c.column_id", ["TABLE_SCHEMA", "TABLE_NAME", "COLUMN_NAME", "DATA_TYPE", "IS_NULLABLE", "ORDINAL_POSITION", "MAX_LENGTH_BYTES", "NUMERIC_PRECISION", "NUMERIC_SCALE"])
            # Independently probe known table shortcuts even if SQL catalog is empty.
            candidates = {(str(r["TABLE_SCHEMA"]), str(r["TABLE_NAME"])) for _, r in catalog.iterrows()}
            for shortcut in consumer_shortcuts or []:
                parts = shortcut["path"].strip("/").split("/")
                if parts[0] == "Tables" and len(parts) in (1, 2) and shortcut.get("target", {}).get("oneLake", {}).get("path", "").strip("/").count("/") >= 1:
                    candidates.add((parts[1] if len(parts) == 2 else "dbo", shortcut["name"]))
            catalog = pd.DataFrame(sorted(candidates), columns=["TABLE_SCHEMA", "TABLE_NAME"])
            if catalog.empty:
                check("SQL tables", "Discovery", "couldn't verify", "No SQL tables found; check shortcut creation and endpoint metadata sync.")
            for _, row in catalog.iterrows():
                schema, name = row["TABLE_SCHEMA"], row["TABLE_NAME"]
                key = f"{schema}.{name}"
                definition = columns[(columns["TABLE_SCHEMA"] == schema) & (columns["TABLE_NAME"] == name)]
                entry = {"name": key, "schema": json.loads(definition.to_json(orient="records"))}
                try:
                    quote = lambda v: "[" + v.replace("]", "]]") + "]"
                    sample = query_frame(connection, "SELECT COUNT_BIG(*) AS SampleRows FROM (SELECT TOP (1) 1 AS Probe FROM " + quote(schema) + "." + quote(name) + ") AS sample")
                    entry.update(read_ok=True, sample_rows_read=int(sample.iloc[0, 0]))
                    check("SQL tables", key, "working", "Readable through consumer SQL endpoint; no source rows displayed.")
                except Exception as exc:
                    entry.update(read_ok=False, error=str(exc))
                    check("SQL tables", key, "broken", str(exc))
                tables.append(entry)
            display(pd.DataFrame(tables)[["name", "read_ok", "sample_rows_read"]] if tables and all("sample_rows_read" in t for t in tables) else pd.DataFrame(tables))
    except Exception as exc:
        problem("Consumer SQL endpoint", exc)

# Map SQL endpoint IDs to lakehouses so either kind of Direct Lake source is recognized.
related_ids = {x["id"] for x in [source, consumer] if x}
if before:
    related_ids.update(x["id"] for x in [before.get("source"), before.get("consumer")] if x)
for lake in [source, consumer]:
    if lake:
        detail = safely(f"Lakehouse properties: {lake['name']}", lambda lake=lake:
            api_get(f"/v1/workspaces/{lake['workspace_id']}/lakehouses/{lake['id']}"))
        endpoint = (detail or {}).get("properties", {}).get("sqlEndpointProperties", {}).get("id")
        if endpoint:
            lake["sql_endpoint_id"] = endpoint
            related_ids.add(endpoint)
if before:
    related_ids.update(x.get("sql_endpoint_id") for x in [before.get("source"), before.get("consumer")] if x and x.get("sql_endpoint_id"))

models = []
for ws, values in items.items():
    for model in [i for i in values if i["type"] == "SemanticModel"]:
        entry = {"workspace_id": ws, "id": model["id"], "name": model["displayName"]}
        try:
            sources = dl.get_direct_lake_sources(dataset=model["id"], workspace=ws)
            sources = json.loads(json.dumps(sources, default=str))
            entry["sources"] = sources
            relevant = any(str(s.get("itemId")) in related_ids for s in sources)
            entry["relevant"] = relevant
            if relevant:
                model_tables = fabric.list_tables(dataset=model["id"], workspace=ws)
                name_col = next((c for c in ["Name", "Table Name"] if c in model_tables.columns), None)
                if not name_col or model_tables.empty:
                    raise ValueError("No model table available for a data-dependent DAX check.")
                table_name = str(model_tables.iloc[0][name_col])
                dax = "EVALUATE ROW(\"HasRows\", NOT ISEMPTY('" + table_name.replace("'", "''") + "'))"
                result = fabric.evaluate_dax(dataset=model["id"], workspace=ws, dax_string=dax)
                entry.update(query_ok=True, tested_table=table_name, result=json.loads(result.to_json(orient="split")))
                check("Models", model["displayName"], "working", f"DAX succeeded for {table_name!r}; remaining tables/measures and freshness are untested.")
            else:
                entry["query_ok"] = None
        except Exception as exc:
            entry.update(query_ok=False, error=str(exc))
            check("Models", model["displayName"], "couldn't verify", str(exc))
        models.append(entry)
display(pd.DataFrame(models))
if not any(m.get("relevant") for m in models):
    check("Models", "Relevant model discovery", "couldn't verify", "No related Direct Lake model identified. Import/DirectQuery models and models outside the configured workspaces are not validated.")
table_checks_current = True


In [ ]:
if not globals().get("table_checks_current", False):
    raise RuntimeError("Run discovery and SQL checks after repair/refresh before saving the report.")

def compare_snapshots(old, new):
    results = []
    def add(area, name, status, detail):
        results.append(dict(area=area, name=name, status=status, detail=detail))
    for role in ["consumer", "source"]:
        previous, current = old.get(role), new.get(role)
        if not previous or not current:
            add("Comparison", role, "couldn't verify", "Identity missing in one snapshot.")
        elif (previous["workspace_id"], previous["id"]) != (current["workspace_id"], current["id"]):
            add("Comparison", role, "broken" if role == "consumer" else "changed", "Item identity changed.")
        else:
            add("Comparison", role, "working", "Item identity preserved.")
    if old.get("shortcuts") is None or new.get("shortcuts") is None:
        add("Comparison", "Shortcut targets", "couldn't verify", "Shortcut inventory unavailable in one snapshot.")
    else:
        a = {(s["path"], s["name"]): s["target"] for s in old["shortcuts"]}
        b = {(s["path"], s["name"]): s["target"] for s in new["shortcuts"]}
        for key in sorted(a.keys() | b.keys()):
            status = "broken" if key not in b else "changed" if key not in a or a[key] != b[key] else "working"
            detail = "Missing shortcut." if key not in b else "Target added/changed." if status == "changed" else "Target unchanged."
            target = b.get(key, {}).get("oneLake", {})
            old_source, new_source = old.get("source"), new.get("source")
            if old_source and new_source and old_source["id"] != new_source["id"] and target.get("itemId") == old_source["id"] and target.get("workspaceId") == old_source["workspace_id"]:
                status, detail = "broken", "Shortcut still targets the legacy source ID."
            add("Comparison", "/".join(key), status, detail)
    a = {t["name"]: t for t in old.get("tables", [])}
    b = {t["name"]: t for t in new.get("tables", [])}
    for name in sorted(a.keys() | b.keys()):
        if name not in b:
            inventory_failed = any(d["operation"] == "Consumer SQL endpoint" for d in new.get("diagnostics", []))
            add("Comparison", name, "couldn't verify" if inventory_failed else "broken", "SQL inventory failed." if inventory_failed else "SQL table missing from after inventory.")
        elif not b[name].get("read_ok"):
            add("Comparison", name, "broken", "After read failed.")
        elif name not in a or a[name].get("schema") != b[name].get("schema"):
            add("Comparison", name, "changed", "Table added or SQL schema changed.")
        else:
            add("Comparison", name, "working", "Readable; SQL schema matches baseline.")
    a = {(m["workspace_id"], m["id"]): m for m in old.get("models", []) if m.get("relevant")}
    b = {(m["workspace_id"], m["id"]): m for m in new.get("models", [])}
    for key, previous in a.items():
        current = b.get(key)
        if not current:
            add("Comparison", previous["name"], "broken", "Baseline model not discovered after migration.")
        elif not current.get("query_ok"):
            add("Comparison", previous["name"], "couldn't verify", "No successful after DAX check.")
        else:
            status = "changed" if previous.get("sources") != current.get("sources") else "working"
            add("Comparison", previous["name"], status, "Model ID preserved; after DAX check succeeded. Inspect source changes and test report freshness.")
    return results

snapshot = {"format_version": 2, "captured_utc": datetime.now(timezone.utc).isoformat(),
            "mode": MODE, "source": source, "consumer": consumer, "shortcuts": consumer_shortcuts,
            "tables": tables, "models": models, "diagnostics": diagnostics,
            "low_latency_flag_observed": LOW_LATENCY_FLAG_OBSERVED}
if MODE == "after" and before:
    checks.extend(compare_snapshots(before, snapshot))
if MODE == "after":
    check("Engine", "Low-latency sync", "working" if LOW_LATENCY_FLAG_OBSERVED is True else "couldn't verify",
          "Flag manually confirmed." if LOW_LATENCY_FLAG_OBSERVED is True else "Confirm the flag in Power Apps; successful reads do not identify the sync engine.")

def save_snapshot():
    if not Path("/lakehouse/default/Files").exists():
        raise ValueError("Attach a separate writable default lakehouse for snapshots.")
    attached_id = str(notebookutils.runtime.context.get("defaultLakehouseId", ""))
    if source and attached_id == source["id"]:
        raise ValueError("The snapshot lakehouse is the source that will be deleted. Attach a separate writable lakehouse.")
    if not consumer or not source:
        raise ValueError("Source/consumer identity is unresolved; review discovery before saving a baseline.")
    snapshot_folder.mkdir(parents=True, exist_ok=True)
    if MODE == "before":
        if not tables or not all(t.get("read_ok") for t in tables):
            raise ValueError("Baseline not saved: table discovery/read checks are incomplete. Fix the endpoint and rerun checks before migration.")
        if baseline_path.exists():
            existing = json.loads(baseline_path.read_text(encoding="utf-8"))
            complete = bool(existing.get("tables")) and all(t.get("read_ok") for t in existing["tables"])
            check("Snapshot", "Baseline", "working" if complete else "couldn't verify", "Existing baseline preserved; this run did not overwrite it. Use a new SNAPSHOT_DIR if the saved baseline is incomplete or belongs to another test.")
            return
        with baseline_path.open("x", encoding="utf-8") as file:
            json.dump(snapshot, file, indent=2, default=str)
        check("Snapshot", "Baseline", "working", str(baseline_path))
    else:
        destination = snapshot_folder / ("after_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
        destination.write_text(json.dumps(snapshot, indent=2, default=str), encoding="utf-8")
        check("Snapshot", "After", "working", str(destination))
safely("Save snapshot", save_snapshot)
report = pd.DataFrame(checks)
display(report)
if not report.empty:
    display(report.groupby("status").size().rename("checks").reset_index())
print("Working means the specific check passed, not that the complete migration is validated.")
if diagnostics:
    display(pd.DataFrame(diagnostics))
